In [31]:
import torch
from torchvision import models
from torch.utils.data import DataLoader
from ImageNet100ValDataset import *


# --- Parámetros ---
target_class_idx = 106  # índice de la clase que quieres contar
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def contador_dirigido(model, val_loader, target_class_idx):
    count_top1_target = 0
    count_top5_target = 0
    total_images = 0

    with torch.no_grad():
        for imgs, labels_idx in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)

            # Filtrar logits sólo a las 100 clases del subset
            filtered_logits = outputs[:, torch.tensor(selected_indices_in_model, device=device)]
            filtered_probs = torch.nn.functional.softmax(filtered_logits, dim=1)

            # Top-1 y Top-5 en el espacio reducido
            preds_top1_rel = filtered_probs.argmax(dim=1)
            top5_rel = torch.topk(filtered_probs, 5, dim=1).indices

            # Mapear de índice relativo (0–99) a índice real de ImageNet
            preds_top1 = torch.tensor(selected_indices_in_model, device=device)[preds_top1_rel]
            top5_preds = torch.tensor(selected_indices_in_model, device=device)[top5_rel]

            # Contar ocurrencias del target real
            for i in range(labels_idx.size(0)):
                if target_class_idx in top5_preds[i]:
                    count_top5_target += 1

            count_top1_target += (preds_top1 == target_class_idx).sum().item()
            total_images += imgs.size(0)

    return count_top1_target / total_images, count_top5_target / total_images

# --- Modelo (ejemplo con ResNet34 preentrenado) ---
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT).to(device).eval()

# --- Dataset (ejemplo con ImageNet o carpeta similar) ---
carpeta = "FGSM_targeted_n01883070"
transform = models.ResNet34_Weights.DEFAULT.transforms()
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)
print(f"Modelo ResNet34")
print(f"Dirigidos a Wombat por FGSM")
print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"\nDirigido a Wombat por RFGSM")


carpeta = "RFGSM_dirigido_target_106"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)

print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"")

carpeta = "PGD_dirigido_target_106_zip (1)"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)

print(f"\nDirigidos a Wombat por PGD")
print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"")


# --- Modelo (ejemplo con ResNet34 preentrenado) ---
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device).eval()

carpeta = "FGSM_targeted_n01883070"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)

print(f"Modelo ResNet50")
print(f"Dirigidos a Wombat por FGSM")
print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"\nDirigido a Wombat por RFGSM")


carpeta = "RFGSM_dirigido_target_106"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)




count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)

print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"")

print(f"\nDirigido a Wombat por PGD")
carpeta = "PGD_dirigido_target_106_zip (1)"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)

print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")
print(f"")

Modelo ResNet34
Dirigidos a Wombat por FGSM
Porcentaje Top-1: 3.80%
Porcentaje Top-5: 8.68%

Dirigido a Wombat por RFGSM
Porcentaje Top-1: 5.52%
Porcentaje Top-5: 12.30%


Dirigidos a Wombat por PGD
Porcentaje Top-1: 36.46%
Porcentaje Top-5: 62.42%

Modelo ResNet50
Dirigidos a Wombat por FGSM
Porcentaje Top-1: 1.88%
Porcentaje Top-5: 5.80%

Dirigido a Wombat por RFGSM
Porcentaje Top-1: 2.40%
Porcentaje Top-5: 7.78%


Dirigido a Wombat por PGD
Porcentaje Top-1: 2.72%
Porcentaje Top-5: 13.16%



In [32]:
# --- Modelo (ejemplo con ResNet34 preentrenado) ---
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT).to(device).eval()

# --- Dataset (ejemplo con ImageNet o carpeta similar) ---
carpeta = "val.X"
val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)
print(f"Modelo ResNet34")
print(f"Eleccion normal de wombat")
print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")

# --- Modelo (ejemplo con ResNet34 preentrenado) ---
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device).eval()

val_ds = ImageNet100ValDataset(carpeta, transform=transform)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)


count_top1_target, count_top5_target = contador_dirigido(model, val_loader, target_class_idx)
print(f"Modelo ResNet50")
print(f"Eleccion normal de wombat")
print(f"Porcentaje Top-1: {100 * count_top1_target:.2f}%")
print(f"Porcentaje Top-5: {100 * count_top5_target:.2f}%")

Modelo ResNet34
Eleccion normal de wombat
Porcentaje Top-1: 0.86%
Porcentaje Top-5: 2.22%
Modelo ResNet50
Eleccion normal de wombat
Porcentaje Top-1: 0.98%
Porcentaje Top-5: 3.30%
